# Trainv vulnerability prediction model

## 1. Imports and config load

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import veps.paths as veps_paths
from veps.config import fetch_config_from_yaml

config = fetch_config_from_yaml()
config.xgboost.device = 'cuda'

print(f'[config] {veps_paths.CONFIG_FILE_PATH}')
print(json.dumps(config.model_dump(), indent=2, default=str))

ARTIFACTS_DIR = (veps_paths.PROJECT_ROOT / 'notebooks' / 'artifacts').resolve()
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'\n[artifacts] {ARTIFACTS_DIR}')

try:
    import torch
    print(f'\n[torch] cuda available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        i = torch.cuda.current_device()
        print(f'[torch] gpu[{i}]: {torch.cuda.get_device_name(i)}')
except Exception as e:
    print(f'[torch] probe skipped ({e})')

## 2. Run DistilBERT predictions on the NVD corpus

In [ ]:
from types import SimpleNamespace
from veps.data.nvd_enrichment import main as predict_bert_main

predict_bert_main(SimpleNamespace(
    single_file=None,
    input_dir=None,
    output_dir=None,
))

## 3. Extract features from the enriched NVD corpus

In [ ]:
from veps.data.feature_extraction import main as extract_features_main

extract_features_main(SimpleNamespace(
    input_dir=None,
    output_file=None,
    latest_only=False,
))

## 4. Build the training set

In [ ]:
from veps.data.exploit_training_builder import create_training_set
from veps.paths import VEPS_TRAINING_SETS

# Set True to skip generation and reuse the most recent training_set_*.csv
# in data/processed/. Handy when iterating on tuning/training only.
USE_LATEST_TRAINING_SET = False

if USE_LATEST_TRAINING_SET:
    candidates = list(VEPS_TRAINING_SETS.glob('training_set_*.csv'))
    if not candidates:
        raise FileNotFoundError(
            f'No training_set_*.csv found in {VEPS_TRAINING_SETS}; '
            'set USE_LATEST_TRAINING_SET=False to generate one.'
        )
    training_file = max(candidates, key=lambda p: p.stat().st_mtime)
    print(f'[trainset] reusing latest: {training_file}')
else:
    pp = config.preprocessing
    training_file = create_training_set(
        window_size=pp.window_size,
        prediction_horizon=pp.prediction_horizon,
        stride=pp.stride_days,
        min_positive_samples=pp.min_positive_samples,
    )
    print(f'\n[trainset] {training_file}')

stats_path = training_file.with_suffix('.stats.json')
if stats_path.exists():
    with open(stats_path) as fh:
        stats = json.load(fh)
    print(f'[stats] {stats_path}')
    print(json.dumps(stats, indent=2))
else:
    print(f'[stats] {stats_path} not found (skipping summary)')


## 5. Train/holdout split

In [ ]:
from veps.exploit_model.data_manager import (
    load_dataset,
    fill_with_predictions,
    resolve_cutoff,
    split_data_for_calibration,
    print_split_stats_with_calibration,
)

df = load_dataset(training_file)
df = fill_with_predictions(
    df,
    use_predicted_cwe=config.use_predictions.cwe,
    use_predicted_cvss=config.use_predictions.cvss,
)

cutoff = resolve_cutoff(df, config)
gap_days = config.preprocessing.resolved_holdout_gap_days()
slice_days = config.calibration.slice_days
print(f'cutoff: {cutoff}   gap_days: {gap_days}   slice_days: {slice_days}')

train_df, calibration_df, test_df = split_data_for_calibration(
    df, cutoff, gap_days=gap_days, calibration_slice_days=slice_days,
)
print_split_stats_with_calibration(
    df, train_df, calibration_df, test_df,
    gap_days=gap_days, slice_days=slice_days,
    target=config.target, label='phase3',
)

## 6. Hyperparameter tuning

In [ ]:
from veps.exploit_model.parameter_tuning import tune_hyperparameters

N_TRIALS = 10
best_params = tune_hyperparameters(
    training_file=training_file,
    n_trials=N_TRIALS,
    device='cuda',
    study_name=config.tune.study_name,
    storage=config.tune.storage,
)

print('\n[best_params]')
print(json.dumps(best_params, indent=2, default=str))


## 7. Final fit on full train split

In [ ]:
from veps.exploit_model.parameter_tuning import train_with_best_params

t0 = time.perf_counter()
pipeline = train_with_best_params(
    training_file=training_file,
    best_params=best_params,
    device='cuda',
)
fit_seconds = time.perf_counter() - t0
print(f'\n[fit] wall-clock: {fit_seconds:.1f}s')

## 8. Holdout metrics

In [ ]:
from veps.exploit_model.calibration import base_pipeline_from, report_metrics

df_full = load_dataset(training_file)
df_full = fill_with_predictions(
    df_full,
    use_predicted_cwe=config.use_predictions.cwe,
    use_predicted_cvss=config.use_predictions.cvss,
)
_train_df, _calibration_df, holdout_df = split_data_for_calibration(
    df_full,
    cutoff,
    gap_days=gap_days,
    calibration_slice_days=slice_days,
)

X_holdout = holdout_df.drop(columns=[config.target])
y_holdout = holdout_df[config.target].to_numpy()

# `pipeline` from section 7 is the deployable: a CalibratedClassifierCV
# when calibration is enabled, or the bare fitted Pipeline otherwise.
# `base_pipeline_from` recovers the underlying fitted Pipeline either way
# so we can compute uncalibrated probabilities for the before/after plot.
base_pipeline = base_pipeline_from(pipeline)
y_proba_raw = base_pipeline.predict_proba(X_holdout)[:, 1]
y_proba_cal = pipeline.predict_proba(X_holdout)[:, 1]

# Alias retained so the drift table cell keeps working.
y_proba = y_proba_cal

report_metrics(
    y_true=y_holdout,
    y_proba=y_proba_raw,
    label='RAW (uncalibrated)',
    year_anchor=holdout_df['window_end'],
)

In [ ]:
report_metrics(
    y_true=y_holdout,
    y_proba=y_proba_cal,
    label=f'{config.calibration.method.upper()} CALIBRATED',
    year_anchor=holdout_df['window_end'],
)

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos_raw, mean_pred_raw = calibration_curve(
    y_holdout, y_proba_raw, n_bins=10, strategy='quantile'
)
frac_pos_cal, mean_pred_cal = calibration_curve(
    y_holdout, y_proba_cal, n_bins=10, strategy='quantile'
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='perfect')
ax.plot(mean_pred_raw, frac_pos_raw, 'o-', label='raw')
ax.plot(mean_pred_cal, frac_pos_cal, 's-', label='calibrated')
ax.set_xlabel('mean predicted probability')
ax.set_ylabel('observed positive rate')
ax.set_title('Reliability diagram: raw vs calibrated (10 quantile bins)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
reliability_path = ARTIFACTS_DIR / 'reliability_before_after.png'
fig.savefig(reliability_path, dpi=150)
plt.show()

## 9. Feature importance grouped by category

In [ ]:
from veps.exploit_model.calibration import base_pipeline_from

base_pipeline = base_pipeline_from(pipeline)
preprocessor = base_pipeline.named_steps['preprocessor']
classifier = base_pipeline.named_steps['classifier']

feature_names = list(preprocessor.get_feature_names_out())
importances = np.asarray(classifier.feature_importances_)
assert len(feature_names) == len(importances), (
    f'feature/importance length mismatch: {len(feature_names)} vs {len(importances)}'
)

OBS_TRIPLE = {'observations_in_window', 'total_count_in_window', 'has_observations'}
MENTION = {'mentions_in_window', 'total_mentions', 'mention_frequency'}

def group_of(name: str) -> str:
    if name.startswith('cwe__'):
        return 'CWE block'
    if name.startswith('cat__'):
        base = name.split('__', 1)[1]
        return 'cna_id' if base == 'cna_id' else 'CVSS axes'
    if name.startswith('passthrough__'):
        base = name.split('__', 1)[1]
        if base.startswith('ref_'):
            return 'reference flags'
        if base.startswith('days_since_'):
            return 'temporal features'
        if base in OBS_TRIPLE:
            return 'observation triple'
        if base in MENTION:
            return 'mention features'
    return 'other'

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
    'group': [group_of(n) for n in feature_names],
})

group_order = [
    'CVSS axes',
    'CWE block',
    'reference flags',
    'mention features',
    'observation triple',
    'cna_id',
    'temporal features',
    'other',
]
grouped = (
    imp_df.groupby('group')['importance'].sum()
          .reindex(group_order).fillna(0.0)
)
if grouped.get('other', 0.0) == 0.0:
    grouped = grouped.drop('other')

print('grouped feature importance (sum within group):')
print(grouped.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(grouped.index, grouped.values)
ax.set_ylabel('sum of feature_importances_')
ax.set_title('XGBoost feature importance by group')
ax.tick_params(axis='x', rotation=30)
for tick in ax.get_xticklabels():
    tick.set_horizontalalignment('right')
fig.tight_layout()
importance_path = ARTIFACTS_DIR / 'importance_by_group.png'
fig.savefig(importance_path, dpi=150)
plt.show()

In [ ]:
print('Top 20 individual features:')
print(
    imp_df.sort_values('importance', ascending=False)
          .head(20)
          .to_string(index=False)
)

## 9b. Permutation importance grouped by category

Gain importance is biased toward high-cardinality features (e.g. `cna_id`).
Permutation importance reshuffles each feature column on a holdout subsample
and measures the resulting drop in PR-AUC — a cardinality-unbiased ranking
to back the sensor-coverage conclusion.

In [ ]:
from sklearn.inspection import permutation_importance

PERM_SAMPLE_ROWS = 50_000
PERM_N_REPEATS = 5
PERM_RANDOM_STATE = config.xgboost.random_state

perm_seed_rng = np.random.default_rng(PERM_RANDOM_STATE)
if len(holdout_df) > PERM_SAMPLE_ROWS:
    sample_idx = perm_seed_rng.choice(len(holdout_df), size=PERM_SAMPLE_ROWS, replace=False)
    perm_X_raw = X_holdout.iloc[sample_idx]
    perm_y = y_holdout[sample_idx]
else:
    perm_X_raw = X_holdout
    perm_y = y_holdout

# Permute on the transformed matrix so the groups line up with the
# gain-importance plot (per CWE-XXX, per CVSS axis, etc.). Permuting the
# raw input would lump every engineered CWE column together under
# `cwe_list`, losing the bucket resolution we want.
perm_X_t = preprocessor.transform(perm_X_raw)

# n_jobs=1: joblib workers would each grab the same GPU model and
# contend; serial predict on a 50k subsample is fast enough on GPU and
# avoids the device-contention question entirely.
t0 = time.perf_counter()
perm_result = permutation_importance(
    classifier,
    perm_X_t,
    perm_y,
    n_repeats=PERM_N_REPEATS,
    scoring='average_precision',
    random_state=PERM_RANDOM_STATE,
    n_jobs=1,
)
perm_seconds = time.perf_counter() - t0
print(f'[perm-importance] rows={len(perm_y):,}  n_repeats={PERM_N_REPEATS}  wall-clock: {perm_seconds:.1f}s')

In [ ]:
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm_result.importances_mean,
    'importance_std': perm_result.importances_std,
    'group': [group_of(n) for n in feature_names],
})

perm_grouped = (
    perm_df.groupby('group')['importance'].sum()
           .reindex(group_order).fillna(0.0)
)
if perm_grouped.get('other', 0.0) == 0.0:
    perm_grouped = perm_grouped.drop('other')

print('grouped permutation importance (sum of mean PR-AUC drop within group):')
print(perm_grouped.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(perm_grouped.index, perm_grouped.values)
ax.set_ylabel('sum of mean PR-AUC drop')
ax.set_title(f'Permutation importance by group (n={len(perm_y):,}, repeats={PERM_N_REPEATS})')
ax.tick_params(axis='x', rotation=30)
for tick in ax.get_xticklabels():
    tick.set_horizontalalignment('right')
fig.tight_layout()
perm_importance_path = ARTIFACTS_DIR / 'permutation_importance_by_group.png'
fig.savefig(perm_importance_path, dpi=150)
plt.show()

In [ ]:
print('Top 20 individual features by permutation importance:')
print(
    perm_df.sort_values('importance', ascending=False)
           .head(20)
           .to_string(index=False)
)

## 10. Calibration-drift check

In [ ]:
drift_df = pd.DataFrame({
    'year': holdout_df['window_end'].dt.year.to_numpy(),
    'y_true': y_holdout,
    'y_proba': y_proba,
})
drift_table = (
    drift_df.groupby('year')
            .agg(rows=('y_true', 'size'),
                 positive_rate=('y_true', 'mean'),
                 mean_pred_proba=('y_proba', 'mean'))
)
drift_table['gap'] = drift_table['mean_pred_proba'] - drift_table['positive_rate']

print('Calibration drift by window_end year (holdout):')
print(drift_table.to_string(float_format=lambda x: f'{x:.4f}'))

## 11. Save the trained pipeline

In [ ]:
from veps.exploit_model.data_manager import save_pipeline

save_pipeline(pipeline)